# Notebook 01: Data Exploration

This notebook downloads one ArXiv paper, parses the PDF, and reports page and token statistics.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd
import tiktoken

repo_root = Path.cwd().resolve()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.ingestion.arxiv_loader import ArxivLoader
from app.ingestion.pdf_parser import PDFParser

In [2]:
query = "1706.03762"
raw_dir = repo_root / "data" / "raw"

loader = ArxivLoader(raw_dir=str(raw_dir))
papers = loader.fetch_papers(query=query, max_results=1)

if not papers:
    raise RuntimeError("No papers were returned by ArxivLoader.")

paper = papers[0]
paper_summary = pd.DataFrame(
    [
        {
            "arxiv_id": paper.get("arxiv_id", ""),
            "title": paper.get("title", ""),
            "author_count": len(paper.get("authors", [])),
            "pdf_path": paper.get("pdf_path", ""),
        }
    ]
)
display(paper_summary)

,arxiv_id,title,author_count,pdf_path
0,2511.05535v1,Future of AI Models: A Computational perspecti...,2,C:\Users\Aymen\Desktop\ArXiv\arxiv-rag-researc...


In [3]:
parser = PDFParser()
parsed = parser.parse(paper["pdf_path"])

encoding = tiktoken.get_encoding("cl100k_base")
page_token_counts = [len(encoding.encode(page_text or "")) for page_text in parsed.pages]

stats = pd.DataFrame(
    [
        {"metric": "page_count", "value": parsed.page_count},
        {"metric": "total_token_count", "value": parsed.token_count},
        {
            "metric": "avg_tokens_per_page",
            "value": round(parsed.token_count / max(parsed.page_count, 1), 2),
        },
        {
            "metric": "min_tokens_per_page",
            "value": min(page_token_counts) if page_token_counts else 0,
        },
        {
            "metric": "max_tokens_per_page",
            "value": max(page_token_counts) if page_token_counts else 0,
        },
    ]
)

display(stats)
display(pd.Series(page_token_counts, name="tokens_per_page").describe())

,metric,value
0,page_count,20.0
1,total_token_count,11984.0
2,avg_tokens_per_page,599.2
3,min_tokens_per_page,42.0
4,max_tokens_per_page,839.0


count     20.000000
mean     599.250000
std      189.400682
min       42.000000
25%      511.000000
50%      606.000000
75%      743.250000
max      839.000000
Name: tokens_per_page, dtype: float64